<a href="https://colab.research.google.com/github/Alan09G/DS2-Group-Project/blob/main/DS2_Group27_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Part 1: Recycle Image Classification

Retrieving, assessing and normalizing the data

In [ ]:
#pip install kagglehub

In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("techsash/waste-classification-data")

print("Path to dataset files:", path)
print("Contents:", os.listdir(path))

In [ ]:
train_dir = os.path.join(path, "DATASET", "TRAIN")
test_dir = os.path.join(path, "DATASET", "TEST")

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED =42

#Extract data from path - returns _PrefetchDataset object (A wrapper for tf.data.Dataset)
#training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

#validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

#testing dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

In [ ]:
print(train_ds.class_names)
#number of batches
print(train_ds.cardinality().numpy())

In [ ]:
#normalize the data (convert values from 0-255 to 0-1)
normalize = tf.keras.layers.Rescaling(1./255)

#iterate through each batch, for every x, y pair => normalize x, leave y the same
train_ds = train_ds.map(lambda x, y: (normalize(x),y))
val_ds = val_ds.map(lambda x, y: (normalize(x),y))
test_ds = test_ds.map(lambda x, y: (normalize(x), y))

In [ ]:
#improve dataset loading performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

display 9 images from the first batch of the train data

In [ ]:
print(type(train_ds))

class_names = ["Organic", "Recyclable"]

for images, labels in train_ds.take(1):
    print(images.shape)
    print(labels.shape)

    plt.figure(figsize=(10, 10))

    for i in range(9):
        plt.subplot(3,3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(class_names[int(labels[i].numpy().item())])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# Random Forest Classifier

images need to be resized to smaller dimensions to decrease the number of features and decrease the computational time to train the model. Images need to be flattened and each pixel is turned into an individual feature.

In [ ]:
#Random Forest
import ydf

#format data for Random Forest Model
def prepare_rf_data(dataset, size=(32, 32)):
  features = []
  labels = []

  for images, batch_labels in dataset:
    #resize images
    images = tf.image.resize(images, size)

    #Flatten each image
    images = tf.reshape(images, [tf.shape(images)[0], -1])

    #convert tensors to numpy arrays
    features.append(images.numpy())
    labels.append(batch_labels.numpy().reshape(-1))

  X = np.concatenate(features)
  y = np.concatenate(labels).astype(int)

  #Convert to named columns for YDF
  data = {
      f"pixel_{i}": X[:, i]
      for i in range(X.shape[1])
  }

  data["class"] = np.where(y == 0, "Organic", "Recyclable")

  return data



# Use ydf's Random Forest model

 Only analyze(time consuming) when needed.

In [ ]:

RUN_ANALYSIS = False  #analyze is time consuming. Only run when needed

#preprocess training data and train
train_rf = prepare_rf_data(train_ds)
rfc_model = ydf.RandomForestLearner(label="class", task=ydf.Task.CLASSIFICATION, num_trees=100).train(train_rf)

#preprocess test data
test_rf = prepare_rf_data(test_ds)

predictions = rfc_model.predict(test_rf)
evalutaion = rfc_model.evaluate(test_rf)

#Anlysie the model (e.g. partial dependence plot, variable importance)
if RUN_ANALYSIS:
  analysis = rfc_model.analyze(test_rf)
  display(analysis)

random forest report

In [ ]:
# Random Forest Evaluation Report
print(f"Test accuracy: {evalutaion.accuracy}")
print("Full evaluation report:")
evalutaion


# Basic CNN

In [ ]:
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(SEED)

cnn_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)

In [ ]:
cnn_test_loss, cnn_test_accuracy = cnn_model.evaluate(test_ds)

print(f"Basic CNN test accuracy: {cnn_test_accuracy:.4f}")
print(f"Basic CNN test loss: {cnn_test_loss:.4f}")

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Test data uses shuffle=False, so labels and predictions stay aligned
y_true = np.concatenate([
    labels.numpy().ravel() for images, labels in test_ds
]).astype(int)

cnn_probabilities = cnn_model.predict(test_ds).ravel()
cnn_predictions = (cnn_probabilities >= 0.5).astype(int)

print(classification_report(
    y_true,
    cnn_predictions,
    labels=[0, 1],
    target_names=["Organic", "Recyclable"],
    digits=4,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    cnn_predictions,
    labels=[0, 1],
    display_labels=["Organic", "Recyclable"],
    cmap="Blues"
)
plt.title("Basic CNN — Test Confusion Matrix")
plt.show()

In [ ]:
epochs = range(1, len(cnn_history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, cnn_history.history["accuracy"], label="Training")
plt.plot(epochs, cnn_history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Basic CNN Accuracy")
plt.xticks(epochs)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, cnn_history.history["loss"], label="Training")
plt.plot(epochs, cnn_history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Basic CNN Loss")
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()

# MobileNetV2

In [ ]:
tf.keras.utils.set_random_seed(SEED)

mobile_base = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights="imagenet"
)

# Keep the pretrained weights fixed
mobile_base.trainable = False

inputs = layers.Input(shape=(128, 128, 3))

#add data aumentation
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

x = augment(inputs)

# Our datasets are already scaled to [0, 1].
# MobileNetV2 needs inputs scaled to [-1, 1].
x = layers.Rescaling(scale=2.0, offset=-1.0)(x)

x = mobile_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

mobile_model = models.Model(inputs, outputs)

mobile_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

mobile_model.summary()

In [ ]:
mobile_early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

mobile_history = mobile_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[mobile_early_stop]
)

In [ ]:
mobile_test_loss, mobile_test_accuracy = mobile_model.evaluate(test_ds)

print(f"MobileNetV2 test accuracy: {mobile_test_accuracy:.4f}")
print(f"MobileNetV2 test loss: {mobile_test_loss:.4f}")

In [ ]:
mobile_probabilities = mobile_model.predict(test_ds).ravel()
mobile_predictions = (mobile_probabilities >= 0.5).astype(int)

print(classification_report(
    y_true,
    mobile_predictions,
    labels=[0, 1],
    target_names=["Organic", "Recyclable"],
    digits=4,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    mobile_predictions,
    labels=[0, 1],
    display_labels=["Organic", "Recyclable"],
    cmap="Blues"
)
plt.title("MobileNetV2 — Test Confusion Matrix")
plt.show()

In [ ]:
epochs = range(1, len(mobile_history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, mobile_history.history["accuracy"], label="Training")
plt.plot(epochs, mobile_history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("MobileNetV2 Accuracy")
plt.xticks(epochs)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, mobile_history.history["loss"], label="Training")
plt.plot(epochs, mobile_history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MobileNetV2 Loss")
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()

# EfficientNetB0
Train the classification head

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, optimizers, callbacks, models

tf.keras.utils.set_random_seed(SEED)

efficient_base = tf.keras.applications.EfficientNetB0(
    input_shape=(128, 128, 3),
    include_top=False,
    weights="imagenet"
)

# Keep the pretrained weights fixed
efficient_base.trainable = False

#Build custom head
inputs = layers.Input(shape=(128, 128, 3))

#add data aumentation
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

x = augment(inputs)

# EfficientNetB0 normalizes internally and needs inputs in [0, 255].
x = layers.Rescaling(255.0)(x)

x = efficient_base(x, training=False)
x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

efficient_model = models.Model(inputs, outputs)

efficient_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

efficient_model.summary()

In [ ]:
efficient_early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

efficient_history = efficient_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[efficient_early_stop]
)

In [ ]:
efficient_test_loss, efficient_test_accuracy = efficient_model.evaluate(test_ds)

print(f"EfficientNetB0 test accuracy: {efficient_test_accuracy:.4f}")
print(f"EfficientNetB0 test loss: {efficient_test_loss:.4f}")

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

efficient_probabilities = efficient_model.predict(test_ds).ravel()
efficient_predictions = (efficient_probabilities >= 0.5).astype(int)

y_true = np.concatenate([
    labels.numpy().ravel() for images, labels in test_ds
]).astype(int)

print(classification_report(
    y_true,
    efficient_predictions,
    labels=[0, 1],
    target_names=["Organic", "Recyclable"],
    digits=4,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    efficient_predictions,
    labels=[0, 1],
    display_labels=["Organic", "Recyclable"],
    cmap="Blues"
)
plt.title("EfficientNetB0 — Test Confusion Matrix")
plt.show()

In [ ]:
epochs = range(1, len(efficient_history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, efficient_history.history["accuracy"], label="Training")
plt.plot(epochs, efficient_history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("EfficientNetB0 Accuracy")
plt.xticks(epochs)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, efficient_history.history["loss"], label="Training")
plt.plot(epochs, efficient_history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("EfficientNetB0 Loss")
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()

# LeNet-5

In [ ]:
tf.keras.utils.set_random_seed(SEED)

lenet_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(6, (5, 5), activation="tanh"),
    layers.AveragePooling2D((2, 2)),

    layers.Conv2D(16, (5, 5), activation="tanh"),
    layers.AveragePooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(120, activation="tanh"),
    layers.Dense(84, activation="tanh"),
    layers.Dense(1, activation="sigmoid")
])

lenet_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lenet_model.summary()

In [ ]:
lenet_early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

lenet_history = lenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[lenet_early_stop]
)

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

y_true = np.concatenate([
    labels.numpy().ravel() for images, labels in test_ds
]).astype(int)


lenet_test_loss, lenet_test_accuracy = lenet_model.evaluate(test_ds)

print(f"LeNet test accuracy: {lenet_test_accuracy:.4f}")
print(f"LeNet test loss: {lenet_test_loss:.4f}")

lenet_probabilities = lenet_model.predict(test_ds).ravel()
lenet_predictions = (lenet_probabilities >= 0.5).astype(int)

print(classification_report(
    y_true,
    lenet_predictions,
    labels=[0, 1],
    target_names=["Organic", "Recyclable"],
    digits=4,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    lenet_predictions,
    labels=[0, 1],
    display_labels=["Organic", "Recyclable"],
    cmap="Blues"
)
plt.title("LeNet — Test Confusion Matrix")
plt.show()

epochs = range(1, len(lenet_history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, lenet_history.history["accuracy"], label="Training")
plt.plot(epochs, lenet_history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("LeNet Accuracy")
plt.xticks(epochs)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, lenet_history.history["loss"], label="Training")
plt.plot(epochs, lenet_history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LeNet Loss")
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()

# ResNet50

In [ ]:
tf.keras.utils.set_random_seed(SEED)

resnet_base = tf.keras.applications.ResNet50(
    input_shape=(128, 128, 3),
    include_top=False,
    weights="imagenet"
)


resnet_base.trainable = False

inputs = layers.Input(shape=(128, 128, 3))

#add data aumentation
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

x = augment(inputs)

x = layers.Rescaling(255.0)(x)
x = tf.keras.applications.resnet50.preprocess_input(x)

x = resnet_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

resnet_model = models.Model(inputs, outputs)

resnet_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

resnet_model.summary()

In [ ]:
resnet_early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

resnet_history = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[resnet_early_stop]
)

In [ ]:
resnet_test_loss, resnet_test_accuracy = resnet_model.evaluate(test_ds)

print(f"ResNet50 test accuracy: {resnet_test_accuracy:.4f}")
print(f"ResNet50 test loss: {resnet_test_loss:.4f}")

resnet_probabilities = resnet_model.predict(test_ds).ravel()
resnet_predictions = (resnet_probabilities >= 0.5).astype(int)

print(classification_report(
    y_true,
    resnet_predictions,
    labels=[0, 1],
    target_names=["Organic", "Recyclable"],
    digits=4,
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    resnet_predictions,
    labels=[0, 1],
    display_labels=["Organic", "Recyclable"],
    cmap="Blues"
)
plt.title("ResNet50 — Test Confusion Matrix")
plt.show()

epochs = range(1, len(resnet_history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, resnet_history.history["accuracy"], label="Training")
plt.plot(epochs, resnet_history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ResNet50 Accuracy")
plt.xticks(epochs)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, resnet_history.history["loss"], label="Training")
plt.plot(epochs, resnet_history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet50 Loss")
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()